# 02 Silver transform

Parsed data → a **modelled, query-ready** layer — typed, deduped, and shaped into a small star schema that gold can aggregate without re-deriving anything.

Three output tables:

- **`dim_cards`** — the modelled card dimension, promoted from `bronze_cards`.
- **`silver_battles`** — already deduped, timestamp parsed to a real `timestamp`, `battle_date` derived for partitioning.
- **`silver_deck_cards`** — the decks **exploded** to one row per card appearance
  (8 cards x 2 sides = 16 rows per battle), **broadcast-joined** to `dim_cards`(ideal for joining a massive fact table and a small dimension table). This is the grain gold aggregates for card pick-rate and win-rate.

**Cleaning vs. validation.** Silver_transform *types and reshapes* but does not *drop* rows. Quarantining bad battles (missing deck, negative trophies, unknown card) is layer 3's job (`silver_quality_checks`), so silver stays a faithful, deterministic projection of bronze and the DQ layer has the full population to measure against.

## Config

In [0]:
from pyspark.sql import functions as F, Window, types as T

CATALOG, SCHEMA = "workspace", "clash"

# Sources (bronze)
BRONZE_BATTLES = f"{CATALOG}.{SCHEMA}.bronze_battles"
BRONZE_CARDS   = f"{CATALOG}.{SCHEMA}.bronze_cards"

# Targets (silver)
DIM_CARDS         = f"{CATALOG}.{SCHEMA}.dim_cards"
SILVER_BATTLES    = f"{CATALOG}.{SCHEMA}.silver_battles"
SILVER_DECK_CARDS = f"{CATALOG}.{SCHEMA}.silver_deck_cards"

## `dim_cards` - dimension table

Deduplicate cards by enforcing one row per `card_id`, and derive an `elixir_cost` column for archetype analysis
(cheap / mid / heavy).

In [0]:
card_w = Window.partitionBy("card_id").orderBy(F.col("bronze_loaded_at").desc())

dim_cards = (spark.table(BRONZE_CARDS)
    .withColumn("rn", F.row_number().over(card_w))
    .filter("rn = 1").drop("rn", "bronze_loaded_at")
    .withColumn("elixir_band", F.when(F.col("elixir_cost") <= 2, "cheap")
                                .when(F.col("elixir_cost") <= 4, "mid")
                                .when(F.col("elixir_cost").isNotNull(), "heavy"))
    .withColumn("silver_built_at", F.current_timestamp()))

dim_cards.write.format("delta").mode("overwrite").saveAsTable(DIM_CARDS)
print(spark.table(DIM_CARDS).count(), "cards in dim_cards")
display(spark.table(DIM_CARDS).orderBy("elixir_cost", "card_id").limit(10))

121 cards in dim_cards


card_id,name,rarity,elixir_cost,max_level,elixir_band,silver_built_at
28000006,Mirror,epic,null,11,null,2026-06-10T10:24:26.332Z
26000010,Skeletons,common,1,16,cheap,2026-06-10T10:24:26.332Z
26000030,Ice Spirit,common,1,16,cheap,2026-06-10T10:24:26.332Z
26000031,Fire Spirit,common,1,16,cheap,2026-06-10T10:24:26.332Z
26000084,Electro Spirit,common,1,16,cheap,2026-06-10T10:24:26.332Z
28000016,Heal Spirit,rare,1,14,cheap,2026-06-10T10:24:26.332Z
26000002,Goblins,common,2,16,cheap,2026-06-10T10:24:26.332Z
26000013,Bomber,common,2,16,cheap,2026-06-10T10:24:26.332Z
26000019,Spear Goblins,common,2,16,cheap,2026-06-10T10:24:26.332Z
26000038,Ice Golem,rare,2,14,cheap,2026-06-10T10:24:26.332Z


## `silver_battles` - typed and partitioned battle fact

Deduplication is already done in bronze_battles, so only typing and deriving here:
- `battle_time` (an ISO string in bronze) becomes a real `timestamp`; `battle_date`
  is derived from it as the **partition key**.
- `crown_diff` enables fine-grained winning analysis for the gold layer

In [0]:
bronze = spark.table(BRONZE_BATTLES)

# Type cast for time and calculate crown difference
silver_battles = (bronze
    .withColumn("battle_time_ts", F.to_timestamp("battle_time"))
    .withColumn("battle_date", F.to_date("battle_time_ts"))
    .withColumn("crown_diff", F.col("player_crowns") - F.col("opponent_crowns"))
    .withColumn("silver_built_at", F.current_timestamp()))

# Drop the raw timestamp strings
silver_battles = silver_battles.drop("battle_time", "battle_time_raw") \
    .withColumnRenamed("battle_time_ts", "battle_time")

# Partition by date
(silver_battles.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("battle_date")
    .saveAsTable(SILVER_BATTLES))

sb = spark.table(SILVER_BATTLES)
print(sb.count(), "battles;", sb.select('battle_date').distinct().count(), "battle dates")
display(sb.limit(10))

318176 battles; 69 battle dates


battle_id,type,is_ladder,game_mode,arena_id,arena_name,player_tag,player_name,player_starting_trophies,player_trophy_change,player_crowns,player_cards,player_deck_hash,opponent_tag,opponent_name,opponent_starting_trophies,opponent_trophy_change,opponent_crowns,opponent_cards,opponent_deck_hash,win,source_player_tag,ingested_at,bronze_loaded_at,battle_time,battle_date,crown_diff,silver_built_at
00babc4ee439b8c961fd19c4c2d729518b578160,PvP,true,Ladder,54000144,Spirit Square,#GVQJPURYY,M4rkit0$,14000,null,3,"List(27000010, 26000018, 26000058, 26000040, 28000026, 27000003, 26000042, 26000055)",50fddb76d5ee9eeefe765c29826f0ef6dfc5f489,#U2LJUVP98,Death gun,14000,null,0,"List(28000008, 26000003, 26000049, 26000028, 27000007, 26000036, 26000041, 26000022)",1b578df99719e4ccb9ad849a1124c4b21410f51e,true,#GVQJPURYY,2026-06-08T02:47:47.888015+00:00,2026-06-10T10:23:27.354Z,2026-05-20T22:21:44.000Z,2026-05-20,3,2026-06-10T10:24:36.063Z
015d20c043833358cff9afa23434a65c20cd20e2,pathOfLegend,false,Ranked1v1_NewArena2,54000149,Legendary Arena,#Y2UL220C2,ShoKaru'❤️,null,30,1,"List(26000055, 26000018, 26000000, 28000000, 28000001, 26000033, 26000042, 28000002)",e6aa87f3db30e8c30a5601ff89719ea3a9138c47,#C002GL82,ぎどら,null,null,0,"List(26000056, 28000015, 26000050, 27000004, 26000040, 26000053, 26000058, 28000026)",b3791212a9c1e75b939a61a025e1f512ef79b50c,true,#Y2UL220C2,2026-06-08T02:30:38.964016+00:00,2026-06-10T10:23:27.354Z,2026-05-20T07:01:02.000Z,2026-05-20,1,2026-06-10T10:24:36.063Z
01fdc32b0507deca3c9253732fec3c1bd5f8fc4f,friendly,false,ClassicDecks_Friendly,54000149,Legendary Arena,#YQ2V8PU2,DamperliKamyon,14000,null,3,"List(26000055, 26000037, 26000041, 26000019, 26000056, 26000032, 26000049, 28000017)",73ffafa19c9636da81e778300485f5a63341522d,#GVPR9PUQL,melg,10253,null,1,"List(26000024, 26000061, 26000044, 28000016, 26000080, 28000011, 28000000, 26000010)",350678f20bcda54259dbe87f036c598498bba2b8,true,#YQ2V8PU2,2026-06-08T03:31:17.709670+00:00,2026-06-10T10:23:27.354Z,2026-05-20T17:58:31.000Z,2026-05-20,2,2026-06-10T10:24:36.063Z
02982229f4ded7e04868dfa1d2754cfaa9e8bc05,pathOfLegend,false,Ranked1v1_NewArena2,54000149,Legendary Arena,#82JYYU8YL,-n1xon-,null,null,1,"List(26000010, 26000072, 26000059, 28000014, 26000031, 28000018, 28000011, 27000000)",e37f10712865b74b3984330979e4070a14adcece,#22YY8R0Y8,Hund,null,30,2,"List(26000055, 26000103, 26000062, 26000020, 28000003, 26000042, 26000007, 28000002)",cf06f6ae104d0f73d431cfabadcf08215f58452b,false,#82JYYU8YL,2026-06-08T02:50:25.218576+00:00,2026-06-10T10:23:27.354Z,2026-05-20T07:49:11.000Z,2026-05-20,-1,2026-06-10T10:24:36.063Z
04abfebd68f5dc367012a65e58aacf094535891e,pathOfLegend,false,Ranked1v1_NewArena2,54000149,Legendary Arena,#9229QYLU,nico,null,null,0,"List(26000011, 26000021, 26000040, 28000008, 26000037, 26000043, 28000000, 26000041)",2f3c02c20faf315ab9cc9126988a880ba39bd4b5,#G8YGVPU,JavierPolanco,null,30,1,"List(26000056, 26000020, 26000044, 26000025, 28000013, 26000011, 28000001, 26000084)",2db9ccbb2a51827190ec6ae9e11a6c647c5fc647,false,#9229QYLU,2026-06-08T02:27:46.666647+00:00,2026-06-10T10:23:27.354Z,2026-05-20T18:22:50.000Z,2026-05-20,-1,2026-06-10T10:24:36.063Z
0630980c8ed4571a6700585308eaeccc72713deb,pathOfLegend,false,Ranked1v1_NewArena2,54000149,Legendary Arena,#2JQ92R9U,habibi,null,30,3,"List(28000008, 26000039, 26000008, 26000006, 26000029, 28000023, 26000013, 26000080)",cdda5ecd02eb261a5324e22cdad95789acb55b5e,#9LJ0JGRYG,❄❃A҉Z҉A҉M҉Y҉❃❄,null,null,1,"List(26000055, 26000018, 26000050, 28000004, 28000001, 28000002, 26000048, 26000033)",c6f17e56024801bda66e01194aad283e57a5d85d,true,#2JQ92R9U,2026-06-08T02:33:43.649543+00:00,2026-06-10T10:23:27.354Z,2026-05-20T07:48:18.000Z,2026-05-20,2,2026-06-10T10:24:36.063Z
07e937052bdbab8da14c8906019695c635c840c7,PvP,true,Ladder,54000001,Goblin Stadium,#20GJ8VRUQP,KEN IS MY DADDY,62,31,3,"List(26000000, 26000001, 26000005, 28000001, 28000000, 26000003, 26000014, 26000018)",1dda8a5053eeabed6fcdb8c1d29aa279797c3168,#VG820Y2YQ,kand,79,-4,

## `silver_deck_cards` — exploded card grain

Each battle carries two decks (player + opponent) of card-id arrays. Here they're
**exploded** to one row per card appearance and **broadcast-joined** to `dim_cards`
for name / rarity / elixir.

Both sides are normalised to the same shape — `side`, the deck owner's `player_tag`,
that side's `crowns`, and a side-specific `won` computed from the crown counts (so the
opponent's win is recorded correctly, not just the subject player's). Gold reads this
table directly: `won` per card row gives card win-rate, row counts give pick-rate.

In [0]:
battles = spark.table(SILVER_BATTLES)

def side_frame(side_label, my, opp):
    """Normalise one side of the battle to card grain. `my`/`opp` are the column
    prefixes ('player' / 'opponent') for the deck owner and their adversary."""
    return (battles.select(
        "battle_id", "battle_date", "battle_time", "arena_id", "is_ladder", "game_mode",
        F.lit(side_label).alias("side"),
        F.col(f"{my}_tag").alias("player_tag"),
        F.col(f"{my}_deck_hash").alias("deck_hash"),
        F.col(f"{my}_crowns").alias("crowns"),
        F.col(f"{opp}_crowns").alias("opponent_crowns"),
        F.col(f"{my}_starting_trophies").alias("starting_trophies"),
        F.col(f"{my}_trophy_change").alias("trophy_change"),
        F.explode(f"{my}_cards").alias("card_id"),
    ))

sides = side_frame("player", "player", "opponent").unionByName(
        side_frame("opponent", "opponent", "player"))

# Side-specific outcome from crown counts (handles the opponent's perspective and draws).
sides = sides.withColumn("won",
    F.when(F.col("crowns").isNull() | F.col("opponent_crowns").isNull(), None)
     .otherwise(F.col("crowns") > F.col("opponent_crowns")))

# Deck size per (battle_id, side) — a window count, useful to the DQ '8 cards' check.
deck_w = Window.partitionBy("battle_id", "side")
sides = sides.withColumn("deck_size", F.count("card_id").over(deck_w))

# Broadcast join: dim_cards is tiny, the fact is large -> ship the dim to every executor
# instead of shuffling the fact. Left join so an unknown card_id survives as null name
# (for the DQ 'every card_id in dim' check) rather than dropping the row.
dims = F.broadcast(spark.table(DIM_CARDS).select(
    "card_id", "name", "rarity", "elixir_cost", "elixir_band"))

silver_deck_cards = (sides.join(dims, on="card_id", how="left")
    .withColumnRenamed("name", "card_name")
    .withColumn("silver_built_at", F.current_timestamp()))

(silver_deck_cards.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("battle_date")
    .saveAsTable(SILVER_DECK_CARDS))

sdc = spark.table(SILVER_DECK_CARDS)
print(sdc.count(), "card-rows =", battles.count(), "battles x ~16")
display(sdc.limit(16))

5316332 card-rows = 318176 battles x ~16


card_id,battle_id,battle_date,battle_time,arena_id,is_ladder,game_mode,side,player_tag,deck_hash,crowns,opponent_crowns,starting_trophies,trophy_change,won,deck_size,card_name,rarity,elixir_cost,elixir_band,silver_built_at
26000011,007f79bd06b33f660ae19ebb44962eca1b7dd6bb,2026-05-20,2026-05-20T06:04:50.000Z,54000143,true,Ladder,player,#9PQYG98U,a300e52dd245c84333ccf09bef2e406b560c65f5,0,3,13380,-26,false,8,Valkyrie,rare,4,mid,2026-06-10T10:24:48.362Z
26000018,007f79bd06b33f660ae19ebb44962eca1b7dd6bb,2026-05-20,2026-05-20T06:04:50.000Z,54000143,true,Ladder,player,#9PQYG98U,a300e52dd245c84333ccf09bef2e406b560c65f5,0,3,13380,-26,false,8,Mini P.E.K.K.A,rare,4,mid,2026-06-10T10:24:48.362Z
26000064,007f79bd06b33f660ae19ebb44962eca1b7dd6bb,2026-05-20,2026-05-20T06:04:50.000Z,54000143,true,Ladder,player,#9PQYG98U,a300e52dd245c84333ccf09bef2e406b560c65f5,0,3,13380,-26,false,8,Firecracker,common,3,mid,2026-06-10T10:24:48.362Z
28000017,007f79bd06b33f660ae19ebb44962eca1b7dd6bb,2026-05-20,2026-05-20T06:04:50.000Z,54000143,true,Ladder,player,#9PQYG98U,a300e52dd245c84333ccf09bef2e406b560c65f5,0,3,13380,-26,false,8,Giant Snowball,common,2,cheap,2026-06-10T10:24:48.362Z
28000003,007f79bd06b33f660ae19ebb44962eca1b7dd6bb,2026-05-20,2026-05-20T06:04:50.000Z,54000143,true,Ladder,player,#9PQYG98U,a300e52dd245c84333ccf09bef2e406b560c65f5,0,3,13380,-26,false,8,Rocket,rare,6,heavy,2026-06-10T10:24:48.362Z
26000041,007f79bd06b33f660ae19ebb44962eca1b7dd6bb,2026-05-20,2026-05-20T06:04:50.000Z,54000143,true,Ladder,player,#9PQYG98U,a300e52dd245c84333ccf09bef2e406b560c65f5,0,3,13380,-26,false,8,Goblin Gang,common,3,mid,2026-06-10T10:24:48.362Z
28000004,007f79bd06b33f660ae19ebb44962eca1b7dd6bb,2026-05-20,2026-05-20T06:04:50.000Z,54000143,true,Ladder,player,#9PQYG98U,a300e52dd245c84333ccf09bef2e406b560c65f5,0,3,13380,-26,false,8,Goblin Barrel,epic,3,mid,2026-06-10T10:24:48.362Z
27000003,007f79bd06b33f660ae19ebb44962eca1b7dd6bb,2026-05-20,2026-05-20T06:04:50.000Z,54000143,true,Ladder,player,#9PQYG98U,a300e52dd245c84333ccf09bef2e406b560c65f5,0,3,13380,-26,false,8,Inferno Tower,rare,5,heavy,2026-06-10T10:24:48.362Z
28000008,00babc4ee439b8c961fd19c4c2d729518b578160,2026-05-20,2026-05-20T22:21:44.000Z,54000144,true,Ladder,opponent,#U2LJUVP98,1b578df99719e4ccb9ad849a1124c4b21410f51e,0,3,14000,null,false,8,Zap,common,2,cheap,2026-06-10T10:24:48.362Z
26000003,00babc4ee439b8c961fd19c4c2d729518b578160,2026-05-20,2026-05-20T22:21:44.000Z,54000144,true,Ladder,opponent,#U2LJUVP98,1b578df99719e4ccb9ad849a1124c4b21410f51e,0,3,14000,null,false,8,Giant,rare,5,heavy,2026-06-10T10:24:48.362Z


## Sanity checks

Not the formal DQ layer, this is just a guard to check out transform's work before moving on to the next layer.

In [0]:
# 1. silver_battles is one row per battle_id.
sb = spark.table(SILVER_BATTLES)
assert sb.count() == sb.select("battle_id").distinct().count(), "silver_battles not unique on battle_id"

# 2. Every deck-card row points back to a real battle (no explode orphans).
sdc = spark.table(SILVER_DECK_CARDS)
orphans = sdc.join(sb.select("battle_id"), on="battle_id", how="left_anti").count()
assert orphans == 0, f"{orphans} deck-card rows with no parent battle"

# 3. Card coverage: how many rows failed to resolve to a dim_cards name?
unknown = sdc.filter(F.col("card_name").isNull()).count()
print(f"deck-card rows: {sdc.count()};  unresolved card_id: {unknown}")
print("distinct decks (player side):",
      sdc.filter("side = 'player'").select("deck_hash").distinct().count())

deck-card rows: 5316332;  unresolved card_id: 0
distinct decks (player side): 48663


## Done

Silver is modelled: `dim_cards`, `silver_battles` (battle grain, partitioned by `battle_date`), and `silver_deck_cards` (card grain, broadcast-joined to the dim). Next: `silver_quality_checks`.